In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")
sample_submission_df = pd.read_csv("sample_submission.csv")

#Display the first few rows of each dataset
display(train_df.head())
display(test_df.head())
display(economic_indicators_df.head())
display(sample_submission_df.head())

#Check missing values
print("Missing Values in Train Data:\n", train_df.isnull().sum())
print("Missing Values in Test Data:\n", test_df.isnull().sum())


,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,Sales
0,Q1,CMP01,5.44,0.05,-0.04,CCC,Buy,South,Metal Fabrication,2.02,1507
1,Q2,CMP01,5.77,0.03,0.00,CCC,Hold,South,Metal Fabrication,2.01,2968
2,Q3,CMP01,4.10,0.06,-0.02,BBB,Buy,South,Metal Fabrication,2.02,3007
3,Q4,CMP01,1.58,0.01,0.02,BBB,Buy,South,Metal Fabrication,1.98,1467
4,Q5,CMP01,2.56,-0.07,0.02,CCC,Buy,South,Metal Fabrication,1.96,2860


,RowID,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio
0,1,Q8,CMP01,5.14,-0.03,-0.01,CCC,Buy,South,Metal Fabrication,1.93
1,2,Q9,CMP01,4.07,0.00,0.00,CCC,Buy,South,Metal Fabrication,1.93
2,3,Q8,CMP02,3.61,0.04,-0.03,A,Sell,West,Infrastructure,1.97
3,4,Q9,CMP02,3.13,0.04,0.01,BBB,Hold,West,Infrastructure,1.93
4,5,Q8,CMP03,NaN,-0.05,-0.01,BB,Buy,East,Infrastructure,0.67


,Month,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,1,67.2,1.538500,55.5,20847.8,57.083078,56.512247,54.628506,56.512247,57.083078
1,2,62.8,1.811579,57.3,20964.3,47.496553,45.454201,47.021588,45.454201,47.496553
2,3,59.4,2.109130,58.8,21115.6,41.697385,39.904398,42.656425,41.280411,41.697385
3,4,65.2,2.777500,59.2,21315.8,48.503429,46.417782,46.417782,48.018395,43.653086
4,5,58.4,2.874286,57.0,21549.3,41.535949,38.379217,39.749903,42.491276,49.843138


,RowID,Sales
0,1,200
1,2,200
2,3,200
3,4,200
4,5,200


Missing Values in Train Data:
 Quarter             0
Company             0
InventoryRatio    106
RevenueGrowth       0
Marketshare         0
Bond rating         0
Stock rating        0
Region              0
Industry            0
QuickRatio          0
Sales               0
dtype: int64
Missing Values in Test Data:
 RowID              0
Quarter            0
Company            0
InventoryRatio    31
RevenueGrowth      0
Marketshare        0
Bond rating        0
Stock rating       0
Region             0
Industry           0
QuickRatio         0
dtype: int64


In [2]:
#Fill missing InventoryRatio using the median for the same company
train_df['InventoryRatio'] = train_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

#Fill remaining missing values using the median for the industry
train_df['InventoryRatio'] = train_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

#Verify if missing values are handled
print("Missing Values in Train Data After Filling:\n", train_df.isnull().sum())
print("Missing Values in Test Data After Filling:\n", test_df.isnull().sum())

Missing Values in Train Data After Filling:
 Quarter           0
Company           0
InventoryRatio    0
RevenueGrowth     0
Marketshare       0
Bond rating       0
Stock rating      0
Region            0
Industry          0
QuickRatio        0
Sales             0
dtype: int64
Missing Values in Test Data After Filling:
 RowID             0
Quarter           0
Company           0
InventoryRatio    0
RevenueGrowth     0
Marketshare       0
Bond rating       0
Stock rating      0
Region            0
Industry          0
QuickRatio        0
dtype: int64


We filled missing values using:

    •    Median InventoryRatio for the same company.
    •    If unavailable, use the median for the industry.

# Merge Economic Indicators
    

*   Convert Quarter to corresponding months.
*   Compute quarterly averages of economic indicators.
*   Merge these indicators with train.csv and test.csv.


In [3]:
#Map Quarter to its corresponding Months
quarter_to_months = {
    "Q1": [1, 2, 3], "Q2": [4, 5, 6], "Q3": [7, 8, 9], "Q4": [10, 11, 12],
    "Q5": [13, 14, 15], "Q6": [16, 17, 18], "Q7": [19, 20, 21],
    "Q8": [22, 23, 24], "Q9": [25, 26, 27], "Q10": [28, 29, 30]
}

#Assign each month to a quarter
economic_indicators_df['Quarter'] = economic_indicators_df['Month'].map(
    lambda x: next((q for q, months in quarter_to_months.items() if x in months), None)
)

#Compute quarterly averages
economic_quarterly_df = economic_indicators_df.groupby("Quarter").mean().reset_index()

#Merge with train and test datasets
train_df = train_df.merge(economic_quarterly_df, on="Quarter", how="left")
test_df = test_df.merge(economic_quarterly_df, on="Quarter", how="left")

#Display merged datasets
display(train_df.head())
display(test_df.head())

,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,...,Month,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,Q1,CMP01,5.44,0.05,-0.04,CCC,Buy,South,Metal Fabrication,2.02,...,2.0,63.133333,1.819736,57.2,20975.900000,48.759005,47.290282,48.102173,47.748953,48.759005
1,Q2,CMP01,5.77,0.03,0.00,CCC,Hold,South,Metal Fabrication,2.01,...,5.0,57.866667,2.947262,56.3,21475.800000,38.324704,36.768411,36.676742,38.124071,39.476987
2,Q3,CMP01,4.10,0.06,-0.02,BBB,Buy,South,Metal Fabrication,2.02,...,8.0,56.100000,3.229186,51.9,21648.566667,36.429408,35.565655,36.163304,35.214299,39.926052
3,Q4,CMP01,1.58,0.01,0.02,BBB,Buy,South,Metal Fabrication,1.98,...,11.0,58.800000,3.999262,48.1,21678.400000,41.830051,40.472376,40.913394,42.286551,41.634372
4,Q5,CMP01,2.56,-0.07,0.02,CCC,Buy,South,Metal Fabrication,1.96,...,14.0,64.600000,3.802861,47.8,21539.333333,51.604474,48.749299,50.452247,51.671275,48.218051


,RowID,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,...,Month,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,1,Q8,CMP01,5.140,-0.03,-0.01,CCC,Buy,South,Metal Fabrication,...,23.0,64.933333,4.421024,49.200000,20846.3,53.884995,51.031343,53.519214,54.592408,56.574052
1,2,Q9,CMP01,4.070,0.00,0.00,CCC,Buy,South,Metal Fabrication,...,26.0,69.900000,3.945484,50.033333,20768.8,61.509561,60.959641,60.197373,59.498423,61.509561
2,3,Q8,CMP02,3.610,0.04,-0.03,A,Sell,West,Infrastructure,...,23.0,64.933333,4.421024,49.200000,20846.3,53.884995,51.031343,53.519214,54.592408,56.574052
3,4,Q9,CMP02,3.130,0.04,0.01,BBB,Hold,West,Infrastructure,...,26.0,69.900000,3.945484,50.033333,20768.8,61.509561,60.959641,60.197373,59.498423,61.509561
4,5,Q8,CMP03,3.475,-0.05,-0.01,BB,Buy,East,Infrastructure,...,23.0,64.933333,4.421024,49.200000,20846.3,53.884995,51.031343,53.519214,54.592408,56.574052


# Feature Engineering
    
1. Encode categorical variables (Bond rating, Stock rating, Region, Industry).

2. Normalize numerical features using StandardScaler.

In [4]:
#Encode categorical variables
categorical_cols = ['Bond rating', 'Stock rating', 'Region', 'Industry']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le

#Select numerical columns for scaling
numerical_cols = ['InventoryRatio', 'RevenueGrowth', 'Marketshare', 'QuickRatio',
                  'Consumer Sentiment', 'Interest Rate', 'PMI', 'Money Supply',
                  'NationalEAI', 'EastEAI', 'WestEAI', 'SouthEAI', 'NorthEAI']

#Apply StandardScaler
scaler = StandardScaler()
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

#Display processed dataset
display(train_df.head())

,Quarter,Company,InventoryRatio,RevenueGrowth,Marketshare,Bond rating,Stock rating,Region,Industry,QuickRatio,...,Month,Consumer Sentiment,Interest Rate,PMI,Money Supply,NationalEAI,EastEAI,WestEAI,SouthEAI,NorthEAI
0,Q1,CMP01,0.139865,0.888745,-2.134776,6,0,2,2,0.685383,...,2.0,0.318906,-2.048025,1.604928,-1.324978,0.351341,0.430248,0.422735,0.274623,0.304332
1,Q2,CMP01,0.212202,0.594459,0.167756,6,1,2,2,0.668162,...,5.0,-0.914346,-0.587094,1.363614,0.387886,-1.077100,-1.124087,-1.165735,-1.039990,-1.126237
2,Q3,CMP01,-0.153865,1.035888,-0.983510,5,0,2,2,0.685383,...,8.0,-1.328032,-0.221806,0.183858,0.979856,-1.336564,-1.301763,-1.237118,-1.437421,-1.057026
3,Q4,CMP01,-0.706253,0.300172,1.319022,5,0,2,2,0.616497,...,11.0,-0.695795,0.775978,-0.835022,1.082078,-0.597223,-0.576922,-0.576716,-0.471459,-0.793735
4,Q5,CMP01,-0.491436,-0.876974,1.319022,6,0,2,2,0.582054,...,14.0,0.662344,0.521502,-0.915460,0.605578,0.740881,0.645780,0.749464,0.810352,0.220959


# Training a Baseline Model
    
1. Train a Random Forest Regressor (as a baseline).
2. Evaluate its performance using MAE (Mean Absolute Error).

In [5]:
#Define features and target variable
X = train_df.drop(columns=['Sales', 'Quarter', 'Company', 'Month'])  # Exclude non-numeric identifiers
y = train_df['Sales']

#Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Train Random Forest Regressor
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

#Predict on validation set
y_val_pred = rf_model.predict(X_val)

#Evaluate model performance
mae = mean_absolute_error(y_val, y_val_pred)
print("Baseline Model MAE:", mae)

Baseline Model MAE: 1471.0714285714287


In [6]:
# Prepare test features (exclude non-numeric identifiers)
X_test = test_df.drop(columns=['RowID', 'Quarter', 'Company', 'Month'])

# Predict sales for test dataset
test_df['Sales'] = rf_model.predict(X_test)

# Prepare submission file
submission_df = test_df[['RowID', 'Sales']].rename(columns={'RowID': 'ID'})

# Save to CSV
submission_df.to_csv("submission_1.csv", index=False)

#Display submission file
display(submission_df.head())

,ID,Sales
0,1,3969.33
1,2,4163.29
2,3,5665.01
3,4,3803.01
4,5,3171.92


In [7]:
#Display submission file
display(submission_df)

,ID,Sales
0,1,3969.33
1,2,4163.29
2,3,5665.01
3,4,3803.01
4,5,3171.92
...,...,...
145,146,3749.05
146,147,3333.97
147,148,4384.50
148,149,3078.21


# Switching from random forest to XGboost

In [8]:
!pip install --upgrade scikit-learn xgboost

In [9]:
#Install XGBoost if not already installed
!pip install xgboost

#Import necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV

In [10]:
#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")
sample_submission_df = pd.read_csv("sample_submission.csv")

#Handle missing values
train_df['InventoryRatio'] = train_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

train_df['InventoryRatio'] = train_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

#Merge Economic Indicators
quarter_to_months = {
    "Q1": [1, 2, 3], "Q2": [4, 5, 6], "Q3": [7, 8, 9], "Q4": [10, 11, 12],
    "Q5": [13, 14, 15], "Q6": [16, 17, 18], "Q7": [19, 20, 21],
    "Q8": [22, 23, 24], "Q9": [25, 26, 27], "Q10": [28, 29, 30]
}

economic_indicators_df['Quarter'] = economic_indicators_df['Month'].map(
    lambda x: next((q for q, months in quarter_to_months.items() if x in months), None)
)

economic_quarterly_df = economic_indicators_df.groupby("Quarter").mean().reset_index()
train_df = train_df.merge(economic_quarterly_df, on="Quarter", how="left")
test_df = test_df.merge(economic_quarterly_df, on="Quarter", how="left")

#Encode categorical variables
categorical_cols = ['Bond rating', 'Stock rating', 'Region', 'Industry']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le

#Scale numerical features
numerical_cols = ['InventoryRatio', 'RevenueGrowth', 'Marketshare', 'QuickRatio',
                  'Consumer Sentiment', 'Interest Rate', 'PMI', 'Money Supply',
                  'NationalEAI', 'EastEAI', 'WestEAI', 'SouthEAI', 'NorthEAI']

scaler = StandardScaler()
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

# Train XGBoost Model with Hyperparameter Tuning

Hyperparameter Optimization Strategy

We’ll tune:

    •    n_estimators: Number of boosting rounds.
    •    max_depth: Maximum tree depth.
    •    learning_rate: Step size shrinkage.
    •    subsample: Sample ratio per boosting round.

In [12]:
#Define features and target
X = train_df.drop(columns=['Sales', 'Quarter', 'Company', 'Month'])
y = train_df['Sales']

#Split into train-validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Define XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

#Set up hyperparameter grid
param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0]
}

#Grid Search with Cross Validation
grid_search = GridSearchCV(xgb_model, param_grid, cv=3, scoring='neg_mean_absolute_error', verbose=2)
grid_search.fit(X_train, y_train)

#Best model from grid search
best_xgb_model = grid_search.best_estimator_

#Predict on validation set
y_val_pred = best_xgb_model.predict(X_val)

#Evaluate performance
mae_xgb = mean_absolute_error(y_val, y_val_pred)
print("XGBoost Model MAE:", mae_xgb)


Fitting 3 folds for each of 81 candidates, totalling 243 fits
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.7; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=0.8; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   0.1s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=100, subsample=1.0; total time=   0.0s
[CV] END learning_rate=0.01, max_depth=3, n_estimators=300, subsample=0.7; total time=   0.1

In [13]:
#Prepare test data
X_test = test_df.drop(columns=['RowID', 'Quarter', 'Company', 'Month'])

#Predict Sales for test dataset
test_df['Sales'] = best_xgb_model.predict(X_test)

#Prepare submission file
submission_df = test_df[['RowID', 'Sales']].rename(columns={'RowID': 'ID'})

#Save submission file
submission_df.to_csv("submission_2.csv", index=False)

#Display submission preview
display(submission_df.head())

,ID,Sales
0,1,2634.245117
1,2,2944.204346
2,3,5779.726562
3,4,2945.043701
4,5,5036.504395


In [14]:
#Display submission full
display(submission_df)

,ID,Sales
0,1,2634.245117
1,2,2944.204346
2,3,5779.726562
3,4,2945.043701
4,5,5036.504395
...,...,...
145,146,3210.823486
146,147,3139.176025
147,148,3972.846924
148,149,2905.541260


# Train Both Models (Random Forest + XGBoost)

In [15]:
#Import necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error

#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")

#Handle missing values in InventoryRatio by filling with the median at Company and Industry level
train_df['InventoryRatio'] = train_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

train_df['InventoryRatio'] = train_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

#Merge Economic Indicators: Convert months to quarters and aggregate economic data
quarter_to_months = {
    "Q1": [1, 2, 3], "Q2": [4, 5, 6], "Q3": [7, 8, 9], "Q4": [10, 11, 12],
    "Q5": [13, 14, 15], "Q6": [16, 17, 18], "Q7": [19, 20, 21],
    "Q8": [22, 23, 24], "Q9": [25, 26, 27], "Q10": [28, 29, 30]
}

economic_indicators_df['Quarter'] = economic_indicators_df['Month'].map(
    lambda x: next((q for q, months in quarter_to_months.items() if x in months), None)
)

economic_quarterly_df = economic_indicators_df.groupby("Quarter").mean().reset_index()

#Merge quarterly economic data with train and test sets
train_df = train_df.merge(economic_quarterly_df, on="Quarter", how="left")
test_df = test_df.merge(economic_quarterly_df, on="Quarter", how="left")

#Encode categorical variables
categorical_cols = ['Bond rating', 'Stock rating', 'Region', 'Industry']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le

#scale numerical features
numerical_cols = ['InventoryRatio', 'RevenueGrowth', 'Marketshare', 'QuickRatio',
                  'Consumer Sentiment', 'Interest Rate', 'PMI', 'Money Supply',
                  'NationalEAI', 'EastEAI', 'WestEAI', 'SouthEAI', 'NorthEAI']

scaler = StandardScaler()
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])

# Model Training

In [19]:
#Define features and target
X = train_df.drop(columns=['Sales', 'Quarter', 'Company', 'Month'])
y = train_df['Sales']

#Split into train-validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Train Random Forest
rf_model = RandomForestRegressor(n_estimators=200, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds_val = rf_model.predict(X_val)

#Train XGBoost
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=300, max_depth=5,
                             learning_rate=0.05, subsample=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds_val = xgb_model.predict(X_val)

In [20]:
#Blend predictions (weighted average)
blend_preds_val = 0.6 * rf_preds_val + 0.4 * xgb_preds_val

#Evaluate blended model
mae_blend = mean_absolute_error(y_val, blend_preds_val)
print("Blended Model MAE:", mae_blend)


Blended Model MAE: 1386.7009261462983


In [21]:
#Prepare test data
X_test = test_df.drop(columns=['RowID', 'Quarter', 'Company', 'Month'])

#Predict sales for test dataset using both models
rf_preds_test = rf_model.predict(X_test)
xgb_preds_test = xgb_model.predict(X_test)

#Blend test predictions
test_df['Sales'] = 0.6 * rf_preds_test + 0.4 * xgb_preds_test

#Prepare submission file
submission_df = test_df[['RowID', 'Sales']].rename(columns={'RowID': 'ID'})

#Save submission file
submission_df.to_csv("submission_3.csv", index=False)

#Display submission preview
display(submission_df)


,ID,Sales
0,1,3764.369964
1,2,4046.366197
2,3,5820.504564
3,4,3517.319827
4,5,3564.263359
...,...,...
145,146,3647.550518
146,147,2940.170719
147,148,4271.083931
148,149,2614.407011


# Intergating LightBGM

In [1]:
#install LightGBM
!pip install lightgbm

In [3]:
pip install "dask[dataframe]"

INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.2/243.2 kB 13.0 MB/s eta 0:00:00


In [4]:
#Import necessary libraries
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.feature_selection import RFECV
from sklearn.model_selection import GridSearchCV

# Reloading and Preprocesing Data


In [5]:
#Load datasets
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
economic_indicators_df = pd.read_csv("EconomicIndicators.csv")

#Handle missing values
train_df['InventoryRatio'] = train_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Company')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

train_df['InventoryRatio'] = train_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))
test_df['InventoryRatio'] = test_df.groupby('Industry')['InventoryRatio'].transform(lambda x: x.fillna(x.median()))

#Merge Economic Indicators
quarter_to_months = {
    "Q1": [1, 2, 3], "Q2": [4, 5, 6], "Q3": [7, 8, 9], "Q4": [10, 11, 12],
    "Q5": [13, 14, 15], "Q6": [16, 17, 18], "Q7": [19, 20, 21],
    "Q8": [22, 23, 24], "Q9": [25, 26, 27], "Q10": [28, 29, 30]
}

economic_indicators_df['Quarter'] = economic_indicators_df['Month'].map(
    lambda x: next((q for q, months in quarter_to_months.items() if x in months), None)
)

economic_quarterly_df = economic_indicators_df.groupby("Quarter").mean().reset_index()
train_df = train_df.merge(economic_quarterly_df, on="Quarter", how="left")
test_df = test_df.merge(economic_quarterly_df, on="Quarter", how="left")

#Encode categorical variables
categorical_cols = ['Bond rating', 'Stock rating', 'Region', 'Industry']
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    test_df[col] = le.transform(test_df[col])
    label_encoders[col] = le

#Scale numerical features
numerical_cols = ['InventoryRatio', 'RevenueGrowth', 'Marketshare', 'QuickRatio',
                  'Consumer Sentiment', 'Interest Rate', 'PMI', 'Money Supply',
                  'NationalEAI', 'EastEAI', 'WestEAI', 'SouthEAI', 'NorthEAI']

scaler = StandardScaler()
train_df[numerical_cols] = scaler.fit_transform(train_df[numerical_cols])
test_df[numerical_cols] = scaler.transform(test_df[numerical_cols])


# Feature Selection (Removing Weak Predictors)

We’ll use Recursive Feature Elimination (RFE) to keep only the most important features.

In [7]:
#Define features and target
X = train_df.drop(columns=['Sales', 'Quarter', 'Company', 'Month'])
y = train_df['Sales']

#Split into train-validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

#Feature selection using Random Forest
rf_feature_selector = RandomForestRegressor(n_estimators=100, random_state=42)
rfe = RFECV(rf_feature_selector, step=1, cv=5, scoring='neg_mean_absolute_error')
rfe.fit(X_train, y_train)

#Keep only selected features
selected_features = X.columns[rfe.support_]
X_train = X_train[selected_features]
X_val = X_val[selected_features]
X_test = test_df[selected_features]

print("Selected Features:", selected_features)

Selected Features: Index(['InventoryRatio', 'RevenueGrowth', 'Marketshare', 'Bond rating',
       'Stock rating', 'Region', 'Industry', 'QuickRatio', 'PMI'],
      dtype='object')


# Train & Tune Models

Fine-tueing RF, XGBoost and LightGBM then blen them together

In [8]:
#Train Random Forest
rf_model = RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds_val = rf_model.predict(X_val)

#Train XGBoost
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=400, max_depth=6, learning_rate=0.04, subsample=0.9, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds_val = xgb_model.predict(X_val)

#Train LightGBM
lgb_model = lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=7, subsample=0.8, random_state=42)
lgb_model.fit(X_train, y_train)
lgb_preds_val = lgb_model.predict(X_val)

#Blend Predictions (Weighted Average)
blend_preds_val = 0.5 * rf_preds_val + 0.3 * xgb_preds_val + 0.2 * lgb_preds_val

#Evaluate blended model
mae_blend = mean_absolute_error(y_val, blend_preds_val)
print("Blended Model MAE:", mae_blend)


[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001224 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 267
[LightGBM] [Info] Number of data points in the train set: 420, number of used features: 9
[LightGBM] [Info] Start training from score 3522.545238
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

In [9]:
#Predict on test set
rf_preds_test = rf_model.predict(X_test)
xgb_preds_test = xgb_model.predict(X_test)
lgb_preds_test = lgb_model.predict(X_test)

#Blend test predictions
test_df['Sales'] = 0.5 * rf_preds_test + 0.3 * xgb_preds_test + 0.2 * lgb_preds_test

#Prepare submission file
submission_df = test_df[['RowID', 'Sales']].rename(columns={'RowID': 'ID'})

#Save submission file
submission_df.to_csv("Shravansubmission_4.csv", index=False)

#Display submission preview
display(submission_df)


,ID,Sales
0,1,3733.963222
1,2,3929.144098
2,3,5865.650724
3,4,3375.800621
4,5,3458.164231
...,...,...
145,146,3192.412846
146,147,2843.470144
147,148,4530.447329
148,149,2483.042140
